# Step 9: fine-tune a detector on real Sentinel-2 imagery

Step E of the Sentinel-2 experiment. Steps 6 to 8 showed that models trained on high-resolution imagery do not work on Sentinel-2, and that resampling the training data to 10 m does not fix it: the resampled detector raised thousands of false alarms per scene. So the detector is now trained on real Sentinel-2 imagery.

A standard pretrained YOLO11 is fine-tuned on eleven Sentinel-2 scenes of the Finnish coast in which the Finnish Environment Institute boxed every vessel (zenodo.org/records/15019034, CC BY 4.0). Eight scenes are for training and three for choosing the checkpoint. The seven test scenes are the same as in steps 6 and 8, so the result goes into the same table.

**Before running: your Copernicus login must be in Colab's secrets** (key icon in the left sidebar: `CDSE_USERNAME` and `CDSE_PASSWORD`, with notebook access switched on). The eleven training scenes, about 1.5 GB, are downloaded to Drive on the first run.

Runtime > Change runtime type > GPU before running. If the session drops, run all cells again: the scenes are kept on Drive and detector training continues from its last epoch.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Settings

In [ ]:
REPO_URL = 'https://github.com/bprtdaniel/vessel-watch.git'
BRANCH = 'main'

RUNS_DIR = '/content/drive/MyDrive/5.Projects/runs'
S2_DIR = '/content/drive/MyDrive/5.Projects/Data/sentinel2_finland'  # downloaded scenes and annotations
CACHE_DIR = '/content/cache'                                         # image chips, rebuilt each session

## Copernicus login

In [ ]:
import os
from google.colab import userdata

os.environ['CDSE_USERNAME'] = userdata.get('CDSE_USERNAME')
os.environ['CDSE_PASSWORD'] = userdata.get('CDSE_PASSWORD')
print('Login found for', os.environ['CDSE_USERNAME'].split('@')[0][:3] + '...')

## Code

In [ ]:
import subprocess

REPO_DIR = '/content/vessel-watch'
if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)

# Colab already ships torch, torchvision, pandas, scikit-learn, matplotlib, pyyaml and opencv
%pip install -q --no-deps -e /content/vessel-watch
%pip install -q ultralytics
%cd /content/vessel-watch
!git log --oneline -1

os.environ['VESSELWATCH_RUNS'] = RUNS_DIR
os.environ['VESSELWATCH_S2'] = S2_DIR
os.environ['VESSELWATCH_CACHE'] = CACHE_DIR

## 1. Fine-tune the detector

First the scenes are downloaded and cut into 320 x 320 pixel chips; three lines report how many chips and vessels each split holds. Then 100 epochs of training. Results go to `RUNS_DIR/yolo11s_finland/`; the last line gives precision, recall and mAP on the test chips.

In [ ]:
!python -m vesselwatch.detect.yolo --config configs/detect/yolo11s_finland.yaml

## 2. Score it on the test scenes, next to the others

The same scoring as in steps 6 and 8, on whole scenes: the fine-tuned detector, the detector trained on resampled data, and the published Finnish model. The detector trained on resampled data also has a `Dock` class, which is ignored, as in step 8.

In [ ]:
!python -m vesselwatch.detect.finland --name step_e --data-dir "$S2_DIR" --detector finetuned="$RUNS_DIR/yolo11s_finland/weights/best.pt" --detector at_10m="$RUNS_DIR/yolo11s_obb_10m/weights/best.pt" --reference --drop-class Dock

## Examples

Labelled vessels in green, each detector's boxes in its own colour. Saved in `RUNS_DIR/sentinel2_step_e_examples/`.

In [ ]:
from pathlib import Path
from IPython.display import Image, display

for path in sorted(Path(RUNS_DIR, 'sentinel2_step_e_examples').glob('*.png'))[:6]:
    print(path.name)
    display(Image(filename=str(path)))